# Retail Sales and Customer Insights

**Purpose:** Explore a public retail dataset and answer business questions. Replace the file path and column mapping after checking your dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
sns.set_theme()


## 1. Load the dataset

Place your CSV at `data/raw/retail_sales.csv`. If your file has another name, update the path below.

In [ ]:
DATA_PATH = "../data/raw/retail_sales.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())


## 2. Understand data quality

In [ ]:
display(df.info())
display(df.describe(include="all").T)
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0])
print("Exact duplicate rows:", df.duplicated().sum())


## 3. Map your dataset columns

Update these values to match the exact column names in your CSV. Leave a value as `None` if that field is unavailable.

In [ ]:
# Example names only — inspect df.columns and edit these mappings.
DATE_COL = "Order Date"
PRODUCT_COL = "Product Name"
CATEGORY_COL = "Category"
QUANTITY_COL = "Quantity"
PRICE_COL = "Unit Price"
SALES_COL = None          # Set to an existing sales/revenue column if present
CUSTOMER_COL = "Customer ID"
REGION_COL = "Region"

print(list(df.columns))


## 4. Clean and prepare data

This section uses only columns that you map correctly. Review all cleaning decisions rather than deleting unusual values automatically.

In [ ]:
clean = df.copy()
clean = clean.drop_duplicates()

if DATE_COL in clean.columns:
    clean[DATE_COL] = pd.to_datetime(clean[DATE_COL], errors="coerce")

for col in [QUANTITY_COL, PRICE_COL, SALES_COL]:
    if col is not None and col in clean.columns:
        clean[col] = pd.to_numeric(clean[col], errors="coerce")

# Derive sales only when both quantity and unit price are available.
if SALES_COL is not None and SALES_COL in clean.columns:
    clean["analysis_sales"] = clean[SALES_COL]
elif QUANTITY_COL in clean.columns and PRICE_COL in clean.columns:
    clean["analysis_sales"] = clean[QUANTITY_COL] * clean[PRICE_COL]
else:
    clean["analysis_sales"] = np.nan

print("Cleaned shape:", clean.shape)
display(clean.head())


## 5. Key performance indicators (KPIs)

In [ ]:
if clean["analysis_sales"].notna().any():
    print("Total sales:", round(clean["analysis_sales"].sum(), 2))
    print("Average transaction sales:", round(clean["analysis_sales"].mean(), 2))
else:
    print("Sales KPI unavailable: map a sales column or both quantity and unit price.")

if QUANTITY_COL in clean.columns:
    print("Total quantity:", clean[QUANTITY_COL].sum())

print("Number of records:", len(clean))
if CUSTOMER_COL in clean.columns:
    print("Unique customers:", clean[CUSTOMER_COL].nunique())


## 6. Product/category performance

In [ ]:
group_col = CATEGORY_COL if CATEGORY_COL in clean.columns else PRODUCT_COL
if group_col in clean.columns and clean["analysis_sales"].notna().any():
    performance = (clean.groupby(group_col, dropna=False)["analysis_sales"]
                   .sum().sort_values(ascending=False))
    display(performance.head(10))
    performance.head(10).sort_values().plot(kind="barh", figsize=(9,5), title=f"Top {group_col} by sales")
    plt.xlabel("Sales")
    plt.tight_layout()
    plt.show()
else:
    print("Map a product/category field and a valid sales field to run this analysis.")


## 7. Monthly sales trend

In [ ]:
if DATE_COL in clean.columns and clean["analysis_sales"].notna().any():
    monthly = (clean.dropna(subset=[DATE_COL])
               .set_index(DATE_COL)["analysis_sales"]
               .resample("MS").sum())
    display(monthly.to_frame("sales").head())
    monthly.plot(figsize=(10,4), marker="o", title="Monthly sales trend")
    plt.ylabel("Sales")
    plt.tight_layout()
    plt.show()
else:
    print("Map a valid date column and sales field to run monthly trend analysis.")


## 8. Write evidence-based business insights

After running the notebook, record only findings supported by your actual outputs:

- Highest revenue product/category:
- Strongest sales month:
- Important customer/region pattern (if available):
- Data-quality limitations:
- Recommended business action:

Do not invent results. Mention the dataset source, date range and limitations in your final report.